In [ ]:
# @title **00. Install dssatspatial package**

%pip install --upgrade --force-reinstall --no-deps git+https://github.com/ssakthi888/DSSATspatial.git

In [ ]:
# @title **01. Global Configuration and Environment Initialization**

import os
import sys
import shutil
import tempfile
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# 1. Global Paths & File Definitions
# Modify these absolute paths as needed to match your local system structure
CUSTOM_TEMP = r"D:\DSSATspatial\temp_s"               # custom temporary folder for DSSAT executions
DSSAT_VERSION = "4.8.5"                               # switch between DSSAT versions "4.8.0" and "4.8.5"

MASTER_XLSX = r"D:\DSSATspatial\notebook_demo\master_matrix_template.xlsx" # master excel containing treatment and management details
MASTER_SHEET = "Sheet1"                               # sheetname containing treatment details
WEATHER_COLUMN = "weather"                            # column name of wth file in excel sheet
SOIL_COLUMN = "soil"                                  # column name of soil id in excel sheet
WTH_FOLDER_PATH = r"D:\DSSATspatial\sample_data\weather" # folder contains all wth files
SOL_FILE_PATH = r"D:\DSSATspatial\sample_data\SOIL.SOL"  # single soil file containing all soilprofiles
FILEX_PATH = r"D:\DSSATspatial\sample_data\IBWA8301.MZX" # management file for maize
SIM_DIR = r"D:\DSSATspatial\notebook_demo\Simulation_Workspace" # output simulation directory where output to be saved

# 2. System and Execution Parameters
MAX_WORKERS = 8
BATCH_SIZE = 100
COLUMNS_TO_KEEP = ["Treatment", "cultivar", "Latitude", "Longitude", "WYEAR", "HWAM"]

# 3. Environment Setup & Library Imports (note: imported after defining the folders)
from DSSATspatial import *

print(f"Temporary directory: {tempfile.gettempdir()}")
print("Environment initialized successfully. Ready for data Loading.")

In [ ]:
# @title **02. Data Loading (Weather, Soils, Cultivars, Treatments)**

# 1. Load Weather Data
stations = load_weather(MASTER_XLSX, MASTER_SHEET, WEATHER_COLUMN, WTH_FOLDER_PATH, MAX_WORKERS)

# 2. Load Soil Profile
soils = load_soil(MASTER_XLSX, MASTER_SHEET, SOIL_COLUMN, SOL_FILE_PATH, MAX_WORKERS)

# 3. Load Default Cultivars
crop_genotype = {
    "IB0063": crop.Maize("IB0063"),
    "IB0060": crop.Maize("IB0060")
}

# 4. Load Base FileX Treatments
base_treatments = load_management_file(FILEX_PATH)

print("Data loading complete. Ready for master execution.")

In [ ]:
# @title **03. Single Treatment Validation Test**

# Execute an isolated simulation to verify Fortran processing
run_t1 = run_dssat(
    row_number=1, 
    xlsx_path=MASTER_XLSX, 
    sheet_name=MASTER_SHEET, 
    sim_dir=SIM_DIR, 
    wth_folder=WTH_FOLDER_PATH, 
    treatments=base_treatments, 
    stations=stations, 
    soils=soils, 
    crops=crop_genotype
)

print(f"Executed Treatment: {run_t1[0]}")
print(f"Execution Status: {'Successful' if run_t1[2] is None else run_t1[2]}")

In [ ]:
# @title **04. Master Execution Pipeline**

# Execute the full batch processing pipeline using the staged in-memory variables
run_spatial_batch(
    xlsx_path=MASTER_XLSX,
    sheet_name=MASTER_SHEET,
    sim_dir=SIM_DIR,
    wth_folder=WTH_FOLDER_PATH,
    treatments=base_treatments,
    stations=stations,
    soils=soils,
    crops=crop_genotype,
    max_workers=MAX_WORKERS,
    batch_size=BATCH_SIZE,
    columns_to_keep=COLUMNS_TO_KEEP
)

print(f"Pipeline Execution Finished. Final compiled dataset is located in: {SIM_DIR}")